# AraBERT v02 baseline reproduction: seeds 42, 43, and 44

Run this notebook in Colab after uploading the project to Drive. Seed 42 is verification-only. Seeds 43 and 44 run for the frozen 10-epoch baseline and resume from their last checkpoints when available.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys
import pandas as pd

project_root = Path('/content/drive/MyDrive/mizan-arabert-sentiment')
os.chdir(project_root)
sys.path.insert(0, str(project_root))

In [ ]:
%pip install -q -r requirements-bonus.txt

In [ ]:
from src.bonus.aggregate import aggregate_runs, run_self_test
from src.bonus.train import run_training
import yaml

CONFIG_PATH = Path('configs/bonus/repro_arabertv02_base_seeds.yaml')
SUMMARY_CSV = Path('checkpoints/bonus/repro_arabertv02_base_seeds_summary.csv')
SUMMARY_MARKDOWN = Path('checkpoints/bonus/repro_arabertv02_base_seeds_summary.md')
RUN_SEED_43 = True
RUN_SEED_44 = True

with CONFIG_PATH.open(encoding='utf-8') as handle:
    config = yaml.safe_load(handle)
output_dirs = {int(seed): Path(path) for seed, path in config['paths']['output_dirs'].items()}

# Verify the existing seed 42 artifacts before any new training begins.
seed42_dir = output_dirs[42]
history_path = seed42_dir / 'history.csv'
required_artifacts = [seed42_dir / 'checkpoint_best.pt', seed42_dir / 'checkpoint_last.pt']
problems = []
if not history_path.is_file():
    problems.append(f'missing {history_path}')
else:
    seed42_history = pd.read_csv(history_path)
    if 'epoch' not in seed42_history.columns:
        problems.append('history.csv has no epoch column')
    elif len(seed42_history) != 10 or sorted(seed42_history['epoch'].astype(int).tolist()) != list(range(1, 11)):
        problems.append(f'history.csv epochs differ from exactly 1-10: {seed42_history.get("epoch", pd.Series(dtype=int)).tolist()}')
for artifact in required_artifacts:
    if not artifact.is_file():
        problems.append(f'missing {artifact}')
if problems:
    raise RuntimeError('Seed 42 verification failed; stop for review: ' + '; '.join(problems))
print('Seed 42 verified: history contains exactly epochs 1-10 and both checkpoints exist.')

In [ ]:
run_self_test()
print('Aggregation self-test passed.')

In [ ]:
# This loop is intentionally restricted to seeds 43 and 44.
for seed, enabled in ((43, RUN_SEED_43), (44, RUN_SEED_44)):
    if not enabled:
        print(f'Seed {seed}: skipped by switch.')
        continue
    seed_config = {key: value.copy() if isinstance(value, dict) else value for key, value in config.items()}
    seed_config['training'] = dict(config['training'])
    seed_config['training']['seed'] = seed
    seed_config['training'].pop('seeds', None)
    seed_config['paths'] = dict(config['paths'])
    seed_config['paths']['output_dir'] = str(output_dirs[seed])
    seed_config['paths'].pop('output_dirs', None)
    history_path = output_dirs[seed] / 'history.csv'
    last_checkpoint = output_dirs[seed] / 'checkpoint_last.pt'
    if history_path.is_file() and last_checkpoint.is_file():
        history = pd.read_csv(history_path)
        epochs_complete = (
            'epoch' in history.columns
            and len(history) == 10
            and sorted(history['epoch'].astype(int).tolist()) == list(range(1, 11))
        )
        if epochs_complete:
            print(f'Seed {seed}: already complete; skipping training.')
            continue
    seed_config_path = Path(f'/content/repro_arabertv02_base_seed{seed}.yaml')
    seed_config_path.write_text(yaml.safe_dump(seed_config, sort_keys=False), encoding='utf-8')
    resume = last_checkpoint.is_file()
    print(f'Seed {seed}: {"resuming" if resume else "starting"} in {output_dirs[seed]}')
    run_training(seed_config_path, output_dir=output_dirs[seed], resume=resume)

In [ ]:
if all((output_dirs[seed] / 'history.csv').is_file() for seed in (42, 43, 44)):
    report = aggregate_runs(
        output_dirs,
        summary_csv=SUMMARY_CSV,
        summary_markdown=SUMMARY_MARKDOWN,
    )
    print(f"Saved {report['summary_csv']}")
    print(f"Saved {report['summary_markdown']}")
else:
    print('Aggregation skipped: one or more seed histories are not available yet.')